In [11]:
# WHY: Python caches imported modules, so edits to src/*.py are ignored until the kernel restarts.
# autoreload re-imports a src file when it changes, so you can edit and re-run without restarting.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import sys; sys.path.append("..")
import pandas as pd
from src.blocking import build_index, prepare_pool

D = "../dataset/dev/"

# WHY only dev files: the full raw tables would waste RAM and are not needed here
s1_dev = pd.read_csv(D + "s1_dev.tsv", sep="\t")
gt_dev = pd.read_csv(D + "gt_dev.tsv", sep="\t")
pool_df = pd.concat([pd.read_csv(D + "s2_dev.tsv", sep="\t"),
                     pd.read_csv(D + "s3_dev.tsv", sep="\t")], ignore_index=True)

# WHY header=None: these id files have no header row
val_ids = set(pd.read_csv(D + "val_ids.txt", header=None)[0])

# WHY the saved pairs: they are exactly what the matcher saw, so "offered" means "in these pairs"
val_pairs = pd.read_pickle(D + "val_pairs.pkl")

# WHY truth as sets: a singleton has no matches, so its set is empty
truth = {s: (set(m.split(",")) if isinstance(m, str) else set())
         for s, m in zip(gt_dev["source1_entity_id"], gt_dev["matched_entity_ids"])}

# WHY build once: indexing and cleaning the pool is the slow part
index = build_index(pool_df)
prepared = prepare_pool(pool_df)

print(len(s1_dev), len(pool_df), len(val_ids), len(val_pairs))
print("keys:", len(index))

100000 467494 20000 577150
keys: 1155909


In [13]:
# WHY a set of pairs: fast lookup to ask "did the matcher ever see this pair?"
offered = set(zip(val_pairs["s1"], val_pairs["cand"]))

# WHY only val entities: blocking is scored on these, never on the 80k training ones
misses = []   # (s1_id, missed_record_id)
for eid in val_ids:
    for t in truth[eid]:
        if (eid, t) not in offered:
            misses.append((eid, t))

total_true = sum(len(truth[e]) for e in val_ids)
print("true matches:", total_true)       # expect 69,455
print("misses:", len(misses))            # expect 7,166
print("share:", round(len(misses) / total_true, 4))   # expect about 0.1032

true matches: 69455
misses: 7166
share: 0.1032


In [14]:
from collections import defaultdict
from src.blocking import get_candidates_topk

# WHY group by S1: the first stage runs once per entity, not once per miss (faster)
by_s1 = defaultdict(list)
for eid, t in misses:
    by_s1[eid].append(t)

# WHY set_index: lets us read an S1 entity's name, address and country by its ID
s1_idx = s1_dev.set_index("entity_id")

cut_by_rerank = []     # in the top-100, lost at rerank
not_in_top100 = []     # blocking never ranked it in the top 100

for eid, lost in by_s1.items():
    row = s1_idx.loc[eid]
    # WHY same settings as build_pairs: cap 1000, top 100, or we would measure a different process
    first = get_candidates_topk(row["business_name"], row["business_address"],
                                row["country"], index, 1000, 100)
    for t in lost:
        (cut_by_rerank if t in first else not_in_top100).append((eid, t))

n = len(misses)
print("cut by rerank (was in top-100):", len(cut_by_rerank), f"({len(cut_by_rerank)/n:.1%})")
print("not in top-100:", len(not_in_top100), f"({len(not_in_top100)/n:.1%})")
print("sum:", len(cut_by_rerank) + len(not_in_top100))   # must equal 7,166

cut by rerank (was in top-100): 854 (11.9%)
not in top-100: 6312 (88.1%)
sum: 7166


In [15]:
from collections import Counter
from src.blocking import all_keys
from src.normalize import norm_name

# WHY set_index: lets us read a missed record's name, address and country by its ID
pool_idx = pool_df.set_index("entity_id")
CAP = 1000   # WHY same as the first stage, or we would label a different process

reasons = Counter()
examples = []   # a few "no shared key: other" cases to read by eye

for eid, t in not_in_top100:
    s1 = s1_idx.loc[eid]
    rec = pool_idx.loc[t]
    k1 = all_keys(s1["business_name"], s1["business_address"], s1["country"])
    k2 = all_keys(rec["business_name"], rec["business_address"], rec["country"])
    shared = k1 & k2

    if not shared:
        # WHY look at the record: it tells us what blocked the match
        if rec["business_address"] != rec["business_address"]:      # NaN check
            reasons["1 no shared key: record has no address"] += 1
        elif norm_name(rec["business_name"]) == "":
            reasons["1 no shared key: record name is native script"] += 1
        else:
            reasons["1 no shared key: other"] += 1
            if len(examples) < 8:
                examples.append((s1["business_name"], s1["business_address"],
                                 rec["business_name"], rec["business_address"]))
    elif all(len(index[k]) > CAP for k in shared):
        reasons["2 shared keys only in blocks bigger than the cap"] += 1
    else:
        reasons["3 usable shared key, ranked below 100"] += 1

for r, c in sorted(reasons.items()):
    print(f"{r}: {c} ({c/len(not_in_top100):.1%})")
print("sum:", sum(reasons.values()))   # must equal 6,312

print()
for s1n, s1a, rn, ra in examples:
    print("S1  :", s1n, "|", s1a)
    print("MISS:", rn, "|", ra, "\n")

1 no shared key: other: 627 (9.9%)
1 no shared key: record has no address: 25 (0.4%)
1 no shared key: record name is native script: 990 (15.7%)
2 shared keys only in blocks bigger than the cap: 1625 (25.7%)
3 usable shared key, ranked below 100: 3045 (48.2%)
sum: 6312

S1  : Rev It Up Diner | W680 Oak Shore Drive, Town Of Fountain Prairie, WI
MISS: Ciraquodrex | W680 OAK SHORE DR, TOWN OF FOUNTAIN PRAIRIE, WI 

S1  : Sash Sheep Ltd | C/O Madan Prasad, Islampur Pokhar Ramlal Lane, Muzaffarpur, Bihar
MISS: Smt sashsheep.com | C/o Maan Prasad, Islampur Pokhar Ramlal Lane, Muzaffarpur, बिहार 

S1  : Ika & Sons Pvt Ltd | 111Jain School Street Mansa, Punjab, Mansa, Punjab
MISS: Gildveo | 111Jain School Street Mansa, Punjab, Mansa, Punjab 

S1  : Dodi Chan Immunopharma LLC | 4013 Miami Avenue, Lorain, OH
MISS: DODICHANIMMUNOPHARMA.COM | 04013 MIAMI AVENUE, LORAIN, OH 

S1  : HN Money | 23 Drummer Boy Way, Lexington, MA
MISS: HN Mobtey | DRUMMER BOY WAY, LEXINGTON, MA 

S1  : Reix | 411 Whiteh

In [16]:
from src.blocking import all_keys
CAP = 1000   # WHY same as the first stage, or we would measure a different process

ranks, key_kind = [], Counter()
for eid, t in not_in_top100:
    s1 = s1_idx.loc[eid]
    rec = pool_idx.loc[t]
    k1 = all_keys(s1["business_name"], s1["business_address"], s1["country"])
    shared = k1 & all_keys(rec["business_name"], rec["business_address"], rec["country"])
    usable = [k for k in shared if len(index[k]) <= CAP]
    if not usable:
        continue   # WHY skip: these are causes 1 and 2, not "ranked below 100"

    # WHY rebuild the scores: get_candidates_topk returns only the set, not the ranking
    scores = defaultdict(float)
    for k in k1:
        ids = index.get(k)
        if ids is None or len(ids) > CAP:
            continue
        for rid in ids:
            scores[rid] += 1.0 / len(ids)

    # WHY count records scoring higher: that is the rank of the missed record
    mine = scores[t]
    ranks.append(1 + sum(1 for v in scores.values() if v > mine))
    # WHY key type: tells us if the match was found through the address or only the name
    key_kind["address key shared" if any(k.startswith("a|") for k in usable)
             else "only name keys shared"] += 1

s = pd.Series(ranks)
print("ranked-below-100 misses:", len(s))   # expect 3,045
for lo, hi in [(101, 200), (201, 300), (301, 500), (501, 1000), (1001, 10**9)]:
    print(f"rank {lo}-{hi if hi < 10**9 else 'up'}: {int(((s >= lo) & (s <= hi)).sum())}")
print(dict(key_kind))

ranked-below-100 misses: 3045
rank 101-200: 537
rank 201-300: 230
rank 301-500: 280
rank 501-1000: 234
rank 1001-up: 83
{'only name keys shared': 2758, 'address key shared': 287}


In [17]:
import time
from src.blocking import get_candidates_topk, rerank

POOL = 300   # WHY 300: the open item from Day 2; was 100 in the current pairs
K = 30       # WHY same k as the saved pairs, so the comparison is fair

val_s1 = s1_dev[s1_dev["entity_id"].isin(val_ids)]
found = total = 0
t0 = time.time()
for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, POOL)
    top = rerank(name, addr, first, prepared, K)
    found += len(top & truth[eid])
    total += len(truth[eid])

print(f"pool={POOL}, k={K}: recall={found/total:.4f}   (pool 100 gave 0.8968)")
print("seconds:", round(time.time() - t0, 1))

pool=300, k=30: recall=0.9124   (pool 100 gave 0.8968)
seconds: 19.8


In [21]:
import gc, time
from src.blocking import build_index, get_candidates_topk, rerank

# WHY delete first: two copies of the index at once would waste RAM
del index; gc.collect()
t0 = time.time()
index = build_index(pool_df)
print("keys:", len(index), "| seconds:", round(time.time() - t0, 1))   # was 1,155,909

found = total = 0
for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, 300)
    top = rerank(name, addr, first, prepared, 30)
    found += len(top & truth[eid])
    total += len(truth[eid])
print(f"after zero-fix, pool=300, k=30: recall={found/total:.4f}   (before: 0.9124)")

keys: 1114321 | seconds: 11.5
after zero-fix, pool=300, k=30: recall=0.9187   (before: 0.9124)


In [19]:
from collections import Counter
from src.normalize import norm_address, norm_name

def nums(addr):
    # WHY same cleaning as address_keys: strip leading zeros, or 0017 vs 17 would count as "different"
    return {t.lstrip("0") or "0" for t in norm_address(addr) if t.isdigit()}

groups = Counter()
left = []   # WHY keep them: to read a few "numbers differ" cases by eye

for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, 300)
    top = rerank(name, addr, first, prepared, 30)
    n1 = nums(addr)
    for t in truth[eid] - top:
        rec = pool_idx.loc[t]
        if rec["business_address"] != rec["business_address"]:        # NaN check
            groups["record has no address"] += 1
        elif norm_name(rec["business_name"]) == "" and not (nums(rec["business_address"]) & n1):
            groups["native-script name, no shared number"] += 1
        else:
            n2 = nums(rec["business_address"])
            if not n1 or not n2:
                groups["no house number on one side"] += 1
            elif n1 & n2:
                groups["house number shared (ranking or cap problem)"] += 1
            else:
                groups["house numbers differ"] += 1
                if len(left) < 8:
                    left.append((addr, rec["business_address"]))

total_left = sum(groups.values())
print("remaining misses:", total_left)
for g, c in groups.most_common():
    print(f"{g}: {c} ({c/total_left:.1%})")

print()
for a, b in left:
    print("S1  :", a)
    print("MISS:", b, "\n")

remaining misses: 6086
house number shared (ranking or cap problem): 1956 (32.1%)
no house number on one side: 1611 (26.5%)
native-script name, no shared number: 920 (15.1%)
house numbers differ: 854 (14.0%)
record has no address: 745 (12.2%)

S1  : 25111 21st Avenue, Unit 22, Phoenix, AZ
MISS: 5111 21ST AVE, PHOENIX, AZ 

S1  : 25111 21st Avenue, Unit 22, Phoenix, AZ
MISS: 5111 21ST AVENUE, PHOENIX, AZ 

S1  : 122 Richdale Road, Needham, MA
MISS: 304 RICHDALE ROAD, NEEDHAM, MA 

S1  : 122 Richdale Road, Needham, MA
MISS: 304 RICHDALE ROAD, NEEDHAM, MA 

S1  : 122 Richdale Road, Needham, MA
MISS: 304- Richdale Road, Needham, Massachusetts 

S1  : 891 Thacker Road, Granger, WA
MISS: 91 Thacker Rd, Granger, Washington 

S1  : OK, Tulsa, 3340 4th Street
MISS: 340 FOURTH SAINT, NULL, TULSA TWNSHIP, OK 

S1  : AR, Springdale, 516 Crutcher Street
MISS: 437 Crutcher St, Springdale, Arkansas 



In [22]:
del index; gc.collect()
t0 = time.time()
index = build_index(pool_df)
print("keys:", len(index), "| build seconds:", round(time.time() - t0, 1))   # was 1,114,321 and 12.9

t0 = time.time()
found = total = 0
for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, 300)
    top = rerank(name, addr, first, prepared, 30)
    found += len(top & truth[eid])
    total += len(truth[eid])
print(f"with street keys, pool=300, k=30: recall={found/total:.4f}   (before: 0.9187)")
print("lookup seconds:", round(time.time() - t0, 1))   # was about 21.6 before the zero fix

keys: 2259243 | build seconds: 15.2
with street keys, pool=300, k=30: recall=0.9548   (before: 0.9187)
lookup seconds: 27.7


In [23]:
import time
import pandas as pd
from src.normalize import norm_name, norm_address
from src.features import pair_features
from src.blocking import get_candidates_topk, rerank

D = "../dataset/dev/"
train_ids = set(pd.read_csv(D + "train_ids.txt", header=None)[0])

def build_pairs(s1_rows, pool=300, k=30):
    rows = []
    for eid, name, addr, country in zip(s1_rows["entity_id"], s1_rows["business_name"],
                                        s1_rows["business_address"], s1_rows["country"]):
        first = get_candidates_topk(name, addr, country, index, 1000, pool)
        top = rerank(name, addr, first, prepared, k)
        # WHY outside the loop: the S1 side is the same for all its candidates
        n1, a1 = norm_name(name), norm_address(addr)
        for cid in top:
            n2, a2 = prepared[cid]
            f = pair_features(n1, a1, n2, a2)
            f["s1"], f["cand"] = eid, cid
            f["label"] = int(cid in truth[eid])   # WHY: 1 only for a true match
            rows.append(f)
    return pd.DataFrame(rows)

t0 = time.time()
train_pairs2 = build_pairs(s1_dev[s1_dev["entity_id"].isin(train_ids)])
val_pairs2 = build_pairs(s1_dev[s1_dev["entity_id"].isin(val_ids)])
print("seconds:", round(time.time() - t0, 1))

# WHY new file names: the old pickles stay as the Day 4 reference
train_pairs2.to_pickle(D + "train_pairs_v2.pkl")
val_pairs2.to_pickle(D + "val_pairs_v2.pkl")

val_true = sum(len(truth[e]) for e in val_ids)
print("train:", train_pairs2.shape, "| positive rate:", round(train_pairs2["label"].mean(), 3))
print("val:", val_pairs2.shape, "| recall ceiling:", round(val_pairs2["label"].sum() / val_true, 4))

seconds: 184.1
train: (2388271, 15) | positive rate: 0.111
val: (597177, 15) | recall ceiling: 0.9548


In [24]:
from src.matcher import train_matcher, predict_matches, macro_f05

model2 = train_matcher(train_pairs2)
pred2 = predict_matches(val_pairs2, model2, 0.70)
print("macro F0.5:", round(macro_f05(pred2, truth, list(val_ids)), 4))   # Day 4 was 0.9248

macro F0.5: 0.9572


In [26]:
import json
from pathlib import Path

OUT = Path("../models/dev")
OUT.mkdir(parents=True, exist_ok=True)

# WHY "_v2" in the names: the Day 4 files stay untouched, so you can show before and after
model2.booster_.save_model(str(OUT / "matcher_dev_v2_lgbm.txt"))

# WHY record the scope: the file alone then explains where the model came from
json.dump({
    "trained_on": "dev sample: 80,000 train entities, 2,388,271 pairs (pool 300, k 30, street keys)",
    "val_macro_f05": 0.9572,
    "recall_ceiling": 0.9548,
    "features": list(model2.feature_name_),
    "threshold": 0.70,
    "one_per_record": True,
}, open(OUT / "matcher_dev_v2_config.json", "w"), indent=2)
print("saved")

saved


In [27]:
from collections import Counter

cnt = Counter()
for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, 300)
    top = rerank(name, addr, first, prepared, 30)
    for t in truth[eid] - top:
        # WHY: prepared[t][0] is the cleaned name, and "" means a native-script name
        native = "native-script name" if prepared[t][0] == "" else "latin name"
        where = "cut by rerank" if t in first else "not in pool of 300"
        cnt[(where, native)] += 1

total = sum(cnt.values())
print("remaining misses:", total)   # expect about 3,100
for (where, native), c in cnt.most_common():
    print(f"{where} | {native}: {c} ({c/total:.1%})")

remaining misses: 3139
cut by rerank | native-script name: 1322 (42.1%)
not in pool of 300 | latin name: 1221 (38.9%)
cut by rerank | latin name: 340 (10.8%)
not in pool of 300 | native-script name: 256 (8.2%)


In [28]:
from rapidfuzz import fuzz

def score_parts(name, addr, cand_ids):
    n1, a1 = norm_name(name), norm_address(addr)
    out = []
    for cid in cand_ids:
        n2, a2 = prepared[cid]
        # WHY None, not 0: "unknown" and "different" must stay apart for the new variant
        ns = fuzz.token_set_ratio(n1, n2) / 100 if n1 and n2 else None
        ad = len(a1 & a2) / len(a1 | a2) if a1 and a2 else None
        out.append((cid, ns, ad))
    return out

def top_old(parts, k):
    # WHY same as rerank in src/blocking.py: missing counts as 0. Must reproduce 0.9548 at k=30
    s = sorted((((ns or 0) + (ad or 0)), cid) for cid, ns, ad in parts)
    return {cid for _, cid in s[::-1][:k]}

def top_new(parts, k):
    def sc(ns, ad):
        if ns is None and ad is None: return 0.0
        # WHY doubled: one part is unknown, so the other must carry the whole score
        if ns is None: return 2 * ad
        if ad is None: return 2 * ns
        return ns + ad
    s = sorted((sc(ns, ad), cid) for cid, ns, ad in parts)
    return {cid for _, cid in s[::-1][:k]}

variants = {"old k=30": (top_old, 30), "old k=60": (top_old, 60),
            "new k=30": (top_new, 30), "new k=60": (top_new, 60)}
found = {v: 0 for v in variants}
total = 0
t0 = time.time()
for eid, name, addr, country in zip(val_s1["entity_id"], val_s1["business_name"],
                                    val_s1["business_address"], val_s1["country"]):
    first = get_candidates_topk(name, addr, country, index, 1000, 300)
    parts = score_parts(name, addr, first)
    total += len(truth[eid])
    for v, (fn, k) in variants.items():
        found[v] += len(fn(parts, k) & truth[eid])

for v in variants:
    print(f"{v}: recall={found[v]/total:.4f}")
print("seconds:", round(time.time() - t0, 1))

old k=30: recall=0.9548
old k=60: recall=0.9602
new k=30: recall=0.9711
new k=60: recall=0.9743
seconds: 30.7


In [29]:
from src.blocking import rerank   # WHY re-import: makes sure build_pairs uses the new rerank

t0 = time.time()
train_pairs3 = build_pairs(s1_dev[s1_dev["entity_id"].isin(train_ids)])
val_pairs3 = build_pairs(s1_dev[s1_dev["entity_id"].isin(val_ids)])
print("seconds:", round(time.time() - t0, 1))

# WHY _v3 names: v1 and v2 stay as reference points
train_pairs3.to_pickle(D + "train_pairs_v3.pkl")
val_pairs3.to_pickle(D + "val_pairs_v3.pkl")

print("val:", val_pairs3.shape, "| recall ceiling:", round(val_pairs3["label"].sum() / val_true, 4))
model3 = train_matcher(train_pairs3)
pred3 = predict_matches(val_pairs3, model3, 0.70)
print("macro F0.5:", round(macro_f05(pred3, truth, list(val_ids)), 4))   # v2 was 0.9572

seconds: 176.8
val: (597177, 15) | recall ceiling: 0.9711
macro F0.5: 0.9577


In [30]:
from src.matcher import FEATURES

val_pairs3["prob"] = model3.predict_proba(val_pairs3[FEATURES])[:, 1]

# WHY sweep again: the threshold was chosen on the old pairs, and the mix of pairs has changed
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = predict_matches(val_pairs3, model3, t)
    print(f"t={t}: macro F0.5={macro_f05(p, truth, list(val_ids)):.4f}")

# WHY label==1 only: these are true matches that blocking DID offer
true_pairs = val_pairs3[val_pairs3["label"] == 1]
dropped = true_pairs[true_pairs["prob"] < 0.70]
print("\ntrue pairs offered:", len(true_pairs), "| dropped below 0.70:", len(dropped))

# WHY name_missing: 1 means a native-script name, where only the address supports the match
for flag, label in [(1, "native-script name"), (0, "latin name")]:
    a = int((true_pairs["name_missing"] == flag).sum())
    b = int((dropped["name_missing"] == flag).sum())
    print(f"{label}: offered {a}, dropped {b} ({b/max(a,1):.1%})")

t=0.3: macro F0.5=0.9508
t=0.4: macro F0.5=0.9540
t=0.5: macro F0.5=0.9563
t=0.6: macro F0.5=0.9576
t=0.7: macro F0.5=0.9577
t=0.8: macro F0.5=0.9524

true pairs offered: 67451 | dropped below 0.70: 3201
native-script name: offered 4140, dropped 830 (20.0%)
latin name: offered 63311, dropped 2371 (3.7%)


In [34]:
# WHY set_index: lets us read the S1 and S2/S3 text for a pair by ID
s1_idx = s1_dev.set_index("entity_id")
pool_idx = pool_df.set_index("entity_id")

def show(df, n=6):
    # WHY random_state: the same examples on every run, so we can talk about them
    for r in df.sample(n, random_state=3).itertuples():
        a, b = s1_idx.loc[r.s1], pool_idx.loc[r.cand]
        print(f"prob={r.prob:.2f} addr_jaccard={r.addr_jaccard:.2f} num_match={r.num_match} num_conflict={r.num_conflict}")
        print("S1  :", a["business_name"], "|", a["business_address"])
        print("MISS:", b["business_name"], "|", b["business_address"], "\n")

print("=== dropped, native-script name ===")
show(dropped[dropped["name_missing"] == 1])
print("=== dropped, latin name ===")
show(dropped[dropped["name_missing"] == 0])

=== dropped, native-script name ===
prob=0.07 addr_jaccard=0.38 num_match=1 num_conflict=0
S1  : Gold Technologies Pvt Ltd | 238 Gf, Tuscan Residency, Tdi City Ii, Sector 111, S.A.S.Nagar (Mohali), Rupnagar, Punjab
MISS: ਗੋਲਡ ਟੈਕਨਾਲੋਜੀਜ਼ ਪ੍ਰਾ. ਲਿ. | No #238 Gf, Rupnagar, S.a.s.nagar (Mohali), PB 

prob=0.50 addr_jaccard=0.45 num_match=1 num_conflict=0
S1  : Arihant Services | D-Block, Maple Apartments Dhakoli, Zirakpur, Mohali, 715, Sas Nagar, Punjab
MISS: ਅਰਿਹੰਤ ਸਰਵਿਸਿਜ਼ | #715, MOHALI, SAS NAGAR, Punjab 

prob=0.08 addr_jaccard=0.38 num_match=0 num_conflict=0
S1  : Baba Digital Infrastructure Private Limited | At & Po. Gozariya, Gidc Plot No. 15, Ta: Mehsana, District: Mehsana, Guajrat, Gozariya, Gujarat
MISS: બાબા ડિજિટલ ઇન્ફ્રાસ્ટ્રક્ચર પ્રાઇવેટ લિમિટેડ | AT & PO. GZOARIYA, GUAJRAT, GOZARIYA, Gujarat 

prob=0.60 addr_jaccard=0.27 num_match=1 num_conflict=0
S1  : Global Vision Tech Limited | 64/31, Krishna Kunj Niwas, Bharati Nt, Waskar Shanjalt Bunglow, Prabhat Road, Erandwane, Pun

In [35]:
from src.features import add_relative_features
from src.matcher import FEATURES, train_matcher, predict_matches, macro_f05

tr = add_relative_features(train_pairs3)
va = add_relative_features(val_pairs3)
NEW = [c for c in tr.columns if c.endswith("_gap") or c.endswith("_rank")
       or c in ("n_num_match", "n_cands")]
F2 = FEATURES + NEW
print("new features:", len(NEW))   # expect 10

model4 = train_matcher(tr, F2)
for t in [0.5, 0.6, 0.7, 0.8]:
    p = predict_matches(va, model4, t, F2)
    print(f"t={t}: macro F0.5={macro_f05(p, truth, list(val_ids)):.4f}")   # v3 was 0.9577 at 0.7

# WHY the same split as before: shows whether the native-script drops shrink
va["prob"] = model4.predict_proba(va[F2])[:, 1]
tp = va[va["label"] == 1]
d = tp[tp["prob"] < 0.70]
for flag, name in [(1, "native-script"), (0, "latin")]:
    print(f"{name}: dropped {int((d['name_missing'] == flag).sum())} of {int((tp['name_missing'] == flag).sum())}")

new features: 10
t=0.5: macro F0.5=0.9576
t=0.6: macro F0.5=0.9593
t=0.7: macro F0.5=0.9608
t=0.8: macro F0.5=0.9591
native-script: dropped 699 of 4140
latin: dropped 2151 of 63311


In [37]:
import json
from pathlib import Path

OUT = Path("../models/dev")
# WHY "_v4" in the names: v1 (Day 4) and v2 stay as reference points
model4.booster_.save_model(str(OUT / "matcher_dev_v4_lgbm.txt"))
json.dump({
    "trained_on": "dev sample: 80,000 train entities, pool 300, k 30, street keys, unknown-aware rerank",
    "val_macro_f05": 0.9608,
    "recall_ceiling": 0.9711,
    "features": F2,
    "threshold": 0.70,
    "one_per_record": True,
}, open(OUT / "matcher_dev_v4_config.json", "w"), indent=2)
print("saved")

saved
